# Task 1: Clean and explore the energy dataset

**Goal.** One clean, documented hourly file (`clean_energy.csv`) that Tasks 2 and 3 can use, and an understanding of how household consumption behaves over the day, week and season.

**Dataset.** UCI *Individual Household Electric Power Consumption* (Hebrail and Berard): one house in France, one reading per minute, December 2006 to November 2010.

The cleaning code lives in `clean_energy.py`, so it can be re-run later on the data logged from the real ESP32 hardware.

In [1]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO = Path.cwd().parent if Path.cwd().name == "energy" else Path.cwd()

UCI_FILE = REPO / "individual+household+electric+power+consumption" / "household_power_consumption.txt"
OUT = REPO / "energy"
OUT.mkdir(exist_ok=True)
assert UCI_FILE.exists(), f"raw file not found: {UCI_FILE}"

MAX_FILL_HOURS = 3
TARGET = "Global_active_power"
SUBS = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
P = TARGET


def read_raw(path):
    """Minute-level data indexed by datetime, every column as float."""
    raw = pd.read_csv(path, sep=";", na_values=["?"], low_memory=False)
    raw.index = pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d/%m/%Y %H:%M:%S")
    raw.index.name = "datetime"
    return raw.drop(columns=["Date", "Time"]).astype(float).sort_index()


def to_hourly(raw, max_fill_hours=MAX_FILL_HOURS):
    """Hourly means; short gaps are forward-filled, long gaps are dropped."""
    hourly = raw.resample("1h").mean()

    isna = hourly[TARGET].isna()
    run_id = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run_id).transform("sum")
    short = isna & (run_len <= max_fill_hours)
    hourly = hourly.where(~short, hourly.ffill())

    stats = {
        "raw_rows": len(raw),
        "raw_missing_rows": int(raw[TARGET].isna().sum()),
        "hours_in_range": len(hourly),
        "hours_without_data": int(isna.sum()),
        "hours_filled": int(short.sum()),
    }
    n_before = len(hourly)
    hourly = hourly.dropna()
    stats["hours_dropped"] = n_before - len(hourly)
    stats["hours_final"] = len(hourly)
    return hourly, stats


def add_features(hourly):
    """Calendar features and the energy not covered by the three sub-meters."""
    out = hourly.copy()
    out["unmetered"] = out[TARGET] * 1000 / 60 - out[SUBS].sum(axis=1)   # Wh per minute
    out["hour"] = out.index.hour
    out["weekday"] = out.index.day_of_week                              # 0 = Monday
    out["month"] = out.index.month
    out["is_weekend"] = (out["weekday"] >= 5).astype(int)
    return out

## 1. Load the raw file

The file is semicolon-separated and uses `?` for missing values.

In [ ]:
raw = read_raw(UCI_FILE)
print("raw:", raw.shape, raw.index.min(), "->", raw.index.max())
print(raw.dtypes.to_string())

miss = raw.isna().sum()
print("\nMissing values per column:")
print(pd.DataFrame({"missing": miss, "pct": (miss / len(raw) * 100).round(2)}).to_string())
print("\nrows with at least one missing value:", int(raw.isna().any(axis=1).sum()),
      "| rows with every column missing:", int(raw.isna().all(axis=1).sum()))

## 2. Where are the missing values?

In [ ]:
nan_min = raw[P].isna()
by_month = nan_min.groupby(raw.index.to_period("M")).sum()

run_id = (nan_min != nan_min.shift()).cumsum()
runs = nan_min.groupby(run_id).agg(["first", "size"])
gaps = runs.loc[runs["first"], "size"].sort_values(ascending=False)
print("missing runs:", len(gaps),
      "| longest (minutes):", gaps.head(5).tolist(),
      "| runs longer than 3 hours:", int((gaps > 180).sum()))

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(by_month.index.to_timestamp(), by_month.values, width=20, color="tab:red")
ax.set(title="Missing minutes per month", ylabel="missing minutes")
fig.tight_layout()
plt.show()

## 3. Clean: hourly means, short gaps filled, long gaps dropped, features added

Decisions:
- Minutes are averaged into hours. An hour without any reading stays missing.
- A missing run of up to 3 hours is forward-filled (a short gap, the last value is a fair guess). A longer run is dropped, because filling it would invent data.
- Features: `hour`, `weekday` (0 = Monday), `month`, `is_weekend`, and `unmetered`, the energy not covered by the three sub-meters (`Global_active_power * 1000 / 60 - Sub_metering_1 - Sub_metering_2 - Sub_metering_3`, in Wh per minute).

In [ ]:
hourly, stats = to_hourly(raw)
df = add_features(hourly)

print(stats)
print(df.shape, df.index.min(), "->", df.index.max())
print("nulls:", int(df.isna().sum().sum()), "| duplicate timestamps:", int(df.index.duplicated().sum()))
print("hours with unmetered < 0:", int((df["unmetered"] < 0).sum()))
print()
print(df.dtypes.to_string())
display(df.head(3))

In [ ]:
assert df.isna().sum().sum() == 0
assert df.index.is_monotonic_increasing and not df.index.has_duplicates
assert all(pd.api.types.is_numeric_dtype(t) for t in df.dtypes)
print("OK: no missing values, sorted unique index, numeric columns")

display(df.describe().T.round(3))

## 4. Explore: day, week and season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for wk, label in [(0, "weekday"), (1, "weekend")]:
    prof = df.loc[df["is_weekend"] == wk, P].groupby(df["hour"]).mean()
    axes[0].plot(prof.index, prof.values, marker="o", label=label)
axes[0].set(title="Daily profile of total power", xlabel="hour of day", ylabel="kW")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

df.groupby("hour")[SUBS + ["unmetered"]].mean().plot(ax=axes[1], marker="o")
axes[1].set(title="Daily profile by sub-meter", xlabel="hour of day", ylabel="Wh per minute")
axes[1].set_xticks(range(0, 24, 2))

fig.tight_layout()
plt.show()

In [ ]:
heat = df.pivot_table(values=P, index="weekday", columns="hour", aggfunc="mean")
heat.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][: len(heat)]

fig, ax = plt.subplots(figsize=(14, 3.8))
sns.heatmap(heat, cmap="YlOrRd", ax=ax, cbar_kws={"label": "kW"})
ax.set(title="Weekly profile: mean power by day of week and hour", xlabel="hour of day", ylabel="")
fig.tight_layout()
plt.show()

In [ ]:
monthly = df[P].resample("MS").mean()
season = df[P].groupby(df["month"]).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly.plot(ax=axes[0], marker="o")
axes[0].set(title="Monthly mean power (first and last months are partial)", xlabel="", ylabel="kW")

season.plot(kind="bar", ax=axes[1], color="tab:green")
axes[1].set(title="Mean power by calendar month (all years)", xlabel="month", ylabel="kW")
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
plt.show()

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation heatmap")
fig.tight_layout()
plt.show()

In [ ]:
by_hour = df.groupby("hour")[P].mean()
wk = df.groupby("is_weekend")[P].mean()
mon = df.groupby("month")[P].mean()
winter = df.loc[df["month"].isin([12, 1, 2]), P].mean()
summer = df.loc[df["month"].isin([6, 7, 8]), P].mean()
total_wh = df[P] * 1000 / 60

print(f"overall mean power           : {df[P].mean():.3f} kW (highest hour {df[P].max():.2f} kW)")
print(f"peak hour of day             : {by_hour.idxmax():02d}:00  ({by_hour.max():.3f} kW)")
print(f"lowest hour of day           : {by_hour.idxmin():02d}:00  ({by_hour.min():.3f} kW)")
print(f"weekday vs weekend mean      : {wk[0]:.3f} vs {wk[1]:.3f} kW  (weekend / weekday = {wk[1] / wk[0]:.2f})")
print(f"winter (Dec-Feb) vs summer (Jun-Aug): {winter:.3f} vs {summer:.3f} kW  (winter / summer = {winter / summer:.2f})")
print(f"highest / lowest calendar month: {mon.idxmax()} ({mon.max():.3f}) / {mon.idxmin()} ({mon.min():.3f})")
print("share of total energy:")
for c in SUBS + ["unmetered"]:
    print(f"   {c:15s} {df[c].mean() / total_wh.mean() * 100:5.1f}%")
print(f"\ncorrelation with {P}:")
print(df.drop(columns=[P]).corrwith(df[P]).round(2).to_string())

## Findings

TODO: write 5 to 8 sentences about what the plots show (daily rhythm, weekday vs weekend, seasonal pattern, which sub-meters drive consumption, what the correlations say, what the missing data looks like).

## 5. Save the deliverables

In [ ]:
CSV = OUT / "clean_energy.csv"
df.to_csv(CSV, lineterminator="\n")
print("saved:", CSV, f"{CSV.stat().st_size / 1e6:.1f} MB")
print("sha256:", hashlib.sha256(CSV.read_bytes()).hexdigest()[:16], "(compare it after a re-run)")

reread = pd.read_csv(CSV, index_col=0, parse_dates=True)
pd.testing.assert_frame_equal(reread, df, check_freq=False, check_dtype=False)
print("re-read file matches the table in memory")

DICT = {
    "Global_active_power": ("kW", "household active power, hourly mean of the minute values"),
    "Global_reactive_power": ("kW", "household reactive power, hourly mean"),
    "Voltage": ("V", "hourly mean voltage"),
    "Global_intensity": ("A", "hourly mean current intensity"),
    "Sub_metering_1": ("Wh per minute", "kitchen (dishwasher, oven, microwave), hourly mean"),
    "Sub_metering_2": ("Wh per minute", "laundry room (washing machine, dryer, refrigerator, lights), hourly mean"),
    "Sub_metering_3": ("Wh per minute", "electric water heater and air conditioner, hourly mean"),
    "unmetered": ("Wh per minute", "Global_active_power * 1000 / 60 minus the three sub-meters: consumption they do not cover"),
    "hour": ("0-23", "hour of day"),
    "weekday": ("0-6", "day of week, 0 = Monday"),
    "month": ("1-12", "calendar month"),
    "is_weekend": ("0 or 1", "1 on Saturday and Sunday"),
}
assert set(DICT) == set(df.columns), set(DICT) ^ set(df.columns)

lines = [
    "# Data dictionary: clean_energy.csv",
    "",
    f"Hourly table with {len(df):,} rows from {df.index.min()} to {df.index.max()}. "
    "The first column is the datetime index (`datetime`). Built by the notebook "
    "`01_clean_explore.ipynb` from the UCI Individual Household Electric Power Consumption file.",
    "",
    "| Column | Unit | Meaning |",
    "|---|---|---|",
]
lines += [f"| `{c}` | {DICT[c][0]} | {DICT[c][1]} |" for c in df.columns]
lines += [
    "",
    "## Cleaning decisions",
    "",
    f"1. The raw file is semicolon-separated and `?` means missing: {stats['raw_missing_rows']:,} of {stats['raw_rows']:,} minute rows have no reading.",
    "2. Date and Time are combined into one datetime index (day first).",
    f"3. Minutes are averaged into hours: {stats['hours_in_range']:,} hours in range, {stats['hours_without_data']:,} of them without any reading. The first and last hours are averages of fewer than 60 minutes.",
    f"4. Gaps of up to {MAX_FILL_HOURS} hours are forward-filled ({stats['hours_filled']:,} hours). Longer gaps are dropped ({stats['hours_dropped']:,} hours), so the time index has gaps there.",
    f"5. Result: {stats['hours_final']:,} hours and no missing values.",
    "6. Added columns: `unmetered`, `hour`, `weekday`, `month`, `is_weekend`.",
]
(OUT / "data_dictionary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("saved:", OUT / "data_dictionary.md")

In [ ]:

raw2 = read_raw(UCI_FILE)
hourly2, stats2 = to_hourly(raw2)
df2 = add_features(hourly2)

tmp_csv = OUT / "_check.csv"
df2.to_csv(tmp_csv, lineterminator="\n")
same = tmp_csv.read_bytes() == CSV.read_bytes()
tmp_csv.unlink()

print("stats identical:", stats2 == stats)
print("re-run output identical to clean_energy.csv:", same)